# Fitch Ratings — RACs (produção)

Notebook próprio (não encaixa nos 3 dispatchers genéricos -- estrutura
própria, `div.frw-column` com tipo+data combinados no mesmo texto). Ver
`ingestores/GERAL/teste_fitch.ipynb` para o notebook de teste da Fase 1
que validou ausência de bloqueio anti-bot, estrutura da listagem e os
dois casos de paywall.

## Escopo, por decisão explícita com base no teste

- **Só RAC (Rating Action Commentary)** -- único tipo com conteúdo
  completo aberto, confirmado no teste (texto completo, tabela de
  ratings, fundamentos, sem nenhum sinal de paywall). **Não tenta
  Research/Insight** -- exige login (botão "Access Report" sem `href`,
  ação via JS) -- mesmo motivo pelo qual O Globo e Estadão já foram
  descartados.
- **Selenium com Chrome headless real**, mesmo motor do teste -- a
  listagem depende de JS pra carregar, e o site não teve bloqueio
  anti-bot confirmado com navegador real.
- **Filtro obrigatório de portfólio antes de salvar**: com ~1.100+
  resultados na busca de RACs, só grava item que mencione alguma empresa
  da lista canônica (`scripts/canonical_entidades.json`) -- sem isso a
  fonte vira ruído puro.

## ⚠️ Bloqueio de infraestrutura conhecido (mesmo padrão de Brazil Journal/PPI)

Este notebook **depende de Chrome + chromedriver instalados via init
script** (`/Volumes/desafio_kinea/utils/selenium/init_selenium2_desafio.sh`,
que copia os binários para `/tmp/chrome/...`). Esse init script está
anexado ao cluster **pessoal** usado para validar este notebook
(`chrisaraujofsz@gmail.com's Selenium Personal Cluster`), mas **não** ao
cluster compartilhado que o Job de produção `Ingest-news-INFRA` usa
(`Cluster Desafio`, `0411-142020-dxdkinpz`) -- e o cluster pessoal não
aceita workload de Job (`does not support jobs workload`), então não dá
pra agendar nada nele.

Ou seja: o código abaixo está pronto e validado manualmente (rodado de
ponta a ponta, ver commit), mas **ainda não está registrado como task do
Job de produção nem em `controle_fontes` como "Coberta e validada em
produção"** -- essas duas etapas ficam pendentes até o init script ser
anexado ao cluster compartilhado (ação em andamento com o admin) e uma
execução real, agendada, ser confirmada. Mesmo tratamento já dado a
Brazil Journal e PPI no CLAUDE.md.

In [0]:
%pip install --quiet selenium==4.15.2 beautifulsoup4 lxml rapidfuzz
dbutils.library.restartPython()

In [0]:
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [0]:
# =============================================================================
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
# =============================================================================
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"

In [0]:
# =============================================================================
# Carrega o mecanismo de match determinístico já existente (lista
# canônica + normalização + match exato/fuzzy com threshold) -- reaproveita
# em vez de reimplementar. Ver comentário completo em mencoes_portfolio()
# logo abaixo sobre a única parte que É nova aqui (como gerar candidatos a
# partir de texto livre, já que este dispatcher não tem estágio de LLM
# extraindo nomes primeiro, diferente do pipeline de riscos de crédito).
#
# Nota: esse arquivo tem um bloco `if __name__ == "__main__":` com um
# teste rápido -- pode imprimir algumas linhas de teste ao carregar via
# %run, é inofensivo (mesmo comportamento já observado em outros %run
# deste projeto).
# =============================================================================
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/extrair_riscos_credito"

In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

SOURCE_ID = "fitch_ratings_racs"
SOURCE_DESCRICAO = "Linked from Fitch Ratings — Rating Action Commentary (Brasil)"
BASE_URL = "https://www.fitchratings.com"
SITE_URL = f"{BASE_URL}/search/?expanded=racs&filter.language=Portuguese&filter.country=Brazil"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)
CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

CHROME_BIN = "/tmp/chrome/chrome-linux/chrome"
CHROMEDRIVER_BIN = "/tmp/chrome/chromedriver_linux64/chromedriver"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

# Só a 1a página (24 itens mais recentes, ordenados por data desc) -- não
# testamos paginação na Fase 1, e o volume observado (~4-5 RACs/dia)
# torna 24 itens uma janela de alguns dias, folgada pro ciclo diário com
# dedup por manifesto. Backfill do histórico completo (1.100+ itens) fica
# fora de escopo por ora.
MIN_CHARS_TEXTO = 200

MESES_EN = {
    "jan": "01", "feb": "02", "mar": "03", "apr": "04", "may": "05", "jun": "06",
    "jul": "07", "aug": "08", "sep": "09", "oct": "10", "nov": "11", "dec": "12",
}
PADRAO_DATA_FITCH = re.compile(r"(\d{1,2}) (\w{3}), (\d{4})")

# Aliases com esse tamanho ou menos, mesmo com match de PALAVRA INTEIRA
# (não substring cru), entram como "suspeito" em vez de "confirmado" --
# mesmo padrão do falso positivo achado na Fase 1 (Rumo Malha Paulista
# batendo via aliases curtos/genéricos como "Edge"/"Radar"/"Moove").
LIMIAR_ALIAS_CURTO = 5

## Helpers

In [0]:
def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception as e:
        print(f"[manifesto] falha ao carregar ({e}); iniciando vazio.")
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


def novo_driver():
    options = Options()
    options.binary_location = CHROME_BIN
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1366,900")
    options.add_argument(f"user-agent={USER_AGENT}")
    service = Service(executable_path=CHROMEDRIVER_BIN)
    return webdriver.Chrome(service=service, options=options)

## Filtro de portfólio -- reaproveita `casar_entidade()` de `extrair_riscos_credito.py`

Não reimplementa a lista canônica nem o critério de match (índice de
aliases normalizados + match exato, fuzzy com threshold 0.85) -- isso já
existe e já foi revisado (`extrair_riscos_credito.py`). A única parte
nova aqui é **como gerar candidatos a partir de texto livre**: esse
pipeline determinístico não tem estágio de LLM extraindo nomes de
entidade primeiro (diferente do pipeline de riscos de crédito), então
fazemos varredura por **palavra inteira** (regex `\b`) de cada alias já
normalizado do índice sobre o texto -- não substring cru. É exatamente
essa mudança (substring cru → palavra inteira) que resolve o falso
positivo encontrado na Fase 1.

In [0]:
def mencoes_portfolio(texto: str) -> dict:
    """Retorna {"confirmados": {canonical_id: nome_principal}, "suspeitos": {...}}.

    "Suspeitos" = bateu por palavra inteira, mas via alias curto/genérico
    (LIMIAR_ALIAS_CURTO) -- não descarta silenciosamente (fica nas notas
    do item salvo, sinalizado pra revisão manual), só não conta sozinho
    como confirmação forte de menção real à empresa.
    """
    texto_normalizado = _normalizar(texto)
    confirmados, suspeitos = {}, {}

    for alias_norm, canonical_id in _INDICE_ALIASES.items():
        if not alias_norm:
            continue
        padrao = r"\b" + re.escape(alias_norm) + r"\b"
        if not re.search(padrao, texto_normalizado):
            continue

        resultado = casar_entidade(alias_norm)  # reusa o match existente, não reimplementa
        nome_principal = next(
            (e["nome_principal"] for e in CANONICAL_ENTIDADES if e["canonical_id"] == resultado["canonical_id"]),
            alias_norm,
        )

        if len(alias_norm) <= LIMIAR_ALIAS_CURTO:
            suspeitos[canonical_id] = nome_principal
        else:
            confirmados[canonical_id] = nome_principal

    return {"confirmados": confirmados, "suspeitos": suspeitos}

## Etapa 1 — Listar RACs (Selenium, primeira página)

In [0]:
def listar_racs(driver) -> list[dict]:
    driver.get(SITE_URL)
    time.sleep(6)

    soup = BeautifulSoup(driver.page_source, "lxml")
    itens = []

    for card in soup.select("div.frw-column"):
        tag_titulo = card.select_one("h3.frw-heading--5 a")
        if not tag_titulo:
            continue

        titulo = tag_titulo.get_text(strip=True)
        url = urllib.parse.urljoin(BASE_URL, tag_titulo["href"])

        data_publicacao = None
        tag_tag = card.select_one(".frw-heading--tag")
        if tag_tag:
            texto_tag = tag_tag.get_text(" ", strip=True)
            if "/" in texto_tag:
                _tipo, data_texto = [t.strip() for t in texto_tag.split("/", 1)]
                m = PADRAO_DATA_FITCH.search(data_texto)
                if m:
                    dia, mes_abrev, ano = m.groups()
                    mes = MESES_EN.get(mes_abrev.lower())
                    if mes:
                        data_publicacao = f"{ano}-{mes}-{dia.zfill(2)}"

        tag_resumo = card.select_one("p")
        resumo = tag_resumo.get_text(" ", strip=True) if tag_resumo else ""

        itens.append({
            "titulo": titulo,
            "url": url,
            "published_at": data_publicacao,
            "resumo": resumo,
        })

    return itens

## Etapa 2 — Abrir o RAC e extrair texto completo

Corta o texto a partir da primeira ocorrência do título (remove o
cabeçalho fixo de navegação/banner de cookies, que é boilerplate igual em
toda página, sem valor informativo).

In [0]:
def extrair_rac(driver, item: dict) -> Optional[str]:
    driver.get(item["url"])
    time.sleep(6)

    texto_completo = driver.find_element(By.TAG_NAME, "body").text
    idx = texto_completo.find(item["titulo"])
    texto = texto_completo[idx:] if idx != -1 else texto_completo
    texto = texto.strip()

    return texto if len(texto) >= MIN_CHARS_TEXTO else None

## Etapa 3 — Salvar no Volume

In [0]:
def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json

## Execução

Fluxo por item: (1) checa menção de portfólio em título+resumo da
listagem, primeiro (barato, evita abrir toda página de detalhe à toa);
(2) se achou algo (confirmado ou suspeito), abre a página de detalhe e
**reconfirma contra o texto completo** (mais confiável que o resumo
truncado da listagem); (3) só salva se a checagem final tiver pelo menos
um match (confirmado ou suspeito -- suspeito sozinho ainda salva, com a
sinalização nas notas, em vez de descartar silenciosamente).

**Limitação conhecida, documentada em vez de escondida**: itens cujo
título+resumo não batem com nenhum alias não têm a página de detalhe
aberta -- se uma menção de portfólio aparecesse só no corpo completo (não
no resumo truncado), esse item não seria pego. Dado o volume (~1.100+
RACs no total, ~24 checados por execução) e o custo de abrir cada
detalhe com Selenium, essa é uma escolha deliberada de custo/cobertura,
não um bug.

In [0]:
resumo_execucao = {}

try:
    ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)
    driver = novo_driver()

    try:
        itens = listar_racs(driver)
    except Exception as e:
        print(f"=== falha ao carregar a listagem: {e} ===")
        driver.quit()
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=f"listagem: {e}")
        itens = None

    if itens is not None:
        itens_novos = [i for i in itens if i["url"] not in ja_processados]
        print(f"{len(itens)} RACs na listagem, {len(itens_novos)} novos.")

        salvos = 0
        for item in itens_novos:
            print(f"\n  [item] {item['titulo'][:100]}")

            texto_busca_listagem = f"{item['titulo']} {item['resumo']}"
            match_listagem = mencoes_portfolio(texto_busca_listagem)

            if not match_listagem["confirmados"] and not match_listagem["suspeitos"]:
                print("    -> sem menção a empresa do portfólio (título+resumo); pulando sem abrir detalhe.")
                ja_processados.add(item["url"])  # já checado, não precisa reabrir em execuções futuras
                continue

            texto_completo = extrair_rac(driver, item)
            if texto_completo is None:
                print("    -> download do detalhe falhou ou texto curto demais; pulando.")
                continue

            match_final = mencoes_portfolio(texto_completo)
            confirmados = match_final["confirmados"] or match_listagem["confirmados"]
            suspeitos = match_final["suspeitos"] or match_listagem["suspeitos"]

            if not confirmados and not suspeitos:
                print("    -> menção da listagem não se confirmou no texto completo; pulando.")
                ja_processados.add(item["url"])
                continue

            if suspeitos:
                print(f"    [REVISAR] correspondência via alias curto/genérico -- possível falso positivo: {list(suspeitos.values())}")

            metadados = {
                "source_id": SOURCE_ID,
                "title": item["titulo"],
                "description": SOURCE_DESCRICAO,
                "url": item["url"],
                "date": HOJE,
                "published_at": item["published_at"],
                "portfolio_matches_confirmados": sorted(confirmados.values()),
                "portfolio_matches_suspeitos": sorted(suspeitos.values()),
            }

            caminho_txt, _ = salvar_artefatos(PASTA_DESTINO, item["titulo"], texto_completo, metadados)
            print(f"    -> salvo em {caminho_txt}")

            ja_processados.add(item["url"])
            salvos += 1
            time.sleep(random.uniform(0.5, 1.2))

        driver.quit()
        salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)
        print(f"\n=== {salvos} RAC(s) nova(s) com menção a portfólio, salva(s) de {len(itens_novos)} novo(s) na listagem. ===")
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=salvos)

except Exception as e:
    print(f"=== ERRO GERAL: {e} ===")
    try:
        driver.quit()
    except Exception:
        pass
    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=str(e))

print("\n=== Fim. ===")